In [ ]:
project_path = "/home/jupyter"
import os
import sys
sys.path.append(project_path)
from google.cloud import bigquery, storage

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

from fintrans_toolbox.src import bq_utils as bq

In [ ]:
client = bigquery.Client()

In [ ]:
# Summarise the data by UK Domestic All Spending
UK_spending_by_country3a = '''SELECT time_period_value, destination_country, spend FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and mcc = 'All' and mcg = 'All' and merchant_channel = 'All' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country = 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value, spend ORDER BY time_period_value, destination_country DESC'''
df_by_country3a = bq.read_bq_table_sql(client, UK_spending_by_country3a)
df_by_country3a.head()

In [ ]:
#Caculate UK Domestic Total Spending

import pandas as pd

# Assuming df_by_country3a is the DataFrame with your data
# Ensure 'time_period_value' is a string type and split it to get the year (assuming 'Q1', 'Q2', etc., are part of the time_period_value)

# Extract the year from the time_period_value (assuming it's in the format like '2023-Q1', '2023-Q2', etc.)
df_by_country3a['year'] = df_by_country3a['time_period_value'].str[:4].astype(int)

# Now group by year and sum the spend for each year
df_yearly_spend = df_by_country3a.groupby('year')['spend'].sum().reset_index()

# Optionally, you can sort the result by year
df_yearly_spend = df_yearly_spend.sort_values(by='year')

# Display the yearly totals
print(df_yearly_spend)

In [ ]:
df_by_country3a.to_csv('UK_spending_by_country3a.csv')

In [ ]:
# Display the yearly totals UK Spent in UK
df_yearly_spend.to_csv('UK_yearly_spend_country3a.csv')

In [ ]:
# Summarise the data by B2B Domestic
UK_spending_by_country3b = '''SELECT time_period_value, destination_country, spend FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and mcc = 'All' and mcg = 'BUSINESS TO BUSINESS' and merchant_channel = 'All' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country = 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value, spend ORDER BY time_period_value, destination_country DESC'''
df_by_country3b = bq.read_bq_table_sql(client, UK_spending_by_country3b)
df_by_country3b = df_by_country3b.rename(columns={'spend': 'domestic_spend_b2b'})
df_by_country3b.head()

# Extract the year from the time_period_value (assuming it's in format like '2023-Q1')
df_by_country3b['year'] = df_by_country3b['time_period_value'].str[:4].astype(int)

# Group by year and sum the online spend for each year
df_yearly_spend_total2 = df_by_country3b.groupby('year')['domestic_spend_b2b'].sum().reset_index()

# Optionally, sort the result by year
df_yearly_spend_total2 = df_yearly_spend_total2.sort_values(by='year')

# Display the result
print(df_yearly_spend_total2)

In [ ]:
#Calculate Household Domestic Total

# Merge the two dataframes on the 'year' column
df_yearly_spend_domestic_household = pd.merge(df_yearly_spend_total2, df_yearly_spend, on='year', how='inner')

# Create a new column for the difference between the two spending values
df_yearly_spend_domestic_household['spend_domestic_household'] = df_yearly_spend_domestic_household['spend'] - df_yearly_spend_total2['domestic_spend_b2b']

# Optionally, keep only the relevant columns
df_yearly_spend_domestic_household = df_yearly_spend_domestic_household[['year', 'spend_domestic_household']]

# Display the resulting table
print(df_yearly_spend_domestic_household)


In [ ]:
# Summarise the data by UK Domestic Online Spending All
UK_spending_by_country3 = '''SELECT time_period_value, destination_country, spend FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and mcg = 'All' and merchant_channel = 'Online' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country = 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value, spend ORDER BY time_period_value, destination_country DESC'''
df_by_country3 = bq.read_bq_table_sql(client, UK_spending_by_country3)
# Rename the 'spend' column to 'online_spend'
df_by_country3 = df_by_country3.rename(columns={'spend': 'online_spend'})
df_by_country3.head()


In [ ]:
import pandas as pd

# Assuming df_by_country3a is the DataFrame with your data
# Ensure 'time_period_value' is a string type and split it to get the year (assuming 'Q1', 'Q2', etc., are part of the time_period_value)

# Extract the year from the time_period_value (assuming it's in the format like '2023-Q1', '2023-Q2', etc.)
df_by_country3['year'] = df_by_country3['time_period_value'].str[:4].astype(int)

# Now group by year and sum the spend for each year
df_yearly_spend1 = df_by_country3.groupby('year')['online_spend'].sum().reset_index()

# Optionally, you can sort the result by year
df_yearly_spend1 = df_yearly_spend1.sort_values(by='year')

# Display the yearly totals
print(df_yearly_spend1)

In [ ]:
df_yearly_spend1.to_csv('UK_yearly_spend_Online.csv')

In [ ]:
import pandas as pd

# Read the online spend CSV
df_online_spend = pd.read_csv('UK_yearly_spend_Online.csv')

# Read the total spend CSV
df_total_spend = pd.read_csv('UK_yearly_spend_country3a.csv')

# Display the first few rows of each DataFrame to check the structure
print(df_online_spend.head())
print(df_total_spend.head())

In [ ]:
# Merge the two DataFrames on 'year'
merged_spend = pd.merge(df_online_spend[['year', 'online_spend']], df_total_spend[['year', 'spend']], on='year', how='inner')

# Display the merged DataFrame to verify
print(merged_spend.head())


In [ ]:
# Calculate the online spend ratio (as a percentage)
merged_spend['online_spend_ratio'] = (merged_spend['online_spend'] / merged_spend['spend']) * 100

# Display the DataFrame with the new ratio column
print(merged_spend[['year', 'online_spend_ratio']])


In [ ]:
# Save the result to a new CSV file - UK Domestic Online Spending Ratio
merged_spend.to_csv('UK_yearly_online_spend_ratio.csv', index=False)

# Display a success message
print("The online spend ratio has been saved to 'UK_yearly_online_spend_ratio.csv'.")


In [ ]:
project_path = "/home/jupyter"
import os
import sys
sys.path.append(project_path)

from google.cloud import bigquery
import importlib
import plotly.express as px

import numpy as np
import pandas as pd
from datetime import datetime

import ft_digital_trade.src.utils.read_data as read_utils
import ft_digital_trade.src.utils.clean_utils as clean_utils
import ft_digital_trade.src.utils.calculation_utils as calc_utils
import ft_digital_trade.src.utils.plot_utils as plot_utils

client = bigquery.Client()

visa_data = read_utils.read_visa(cardholder_origin = "uk", cardholders_location = "uk", spend_location = "uk")

visa = calc_utils.calculate_visa(visa_data)
visa = clean_utils.rename_columns(df = visa, suffix = '_spoc')

global_cards = read_utils.read_global_cards()
global_cards = clean_utils.clean_global(global_cards)
global_cards = calc_utils.calculate_global(global_cards, 'card')

global_spend = read_utils.read_global_spend()
global_spend = clean_utils.clean_global(global_spend)
global_spend = calc_utils.calculate_global(global_spend, 'spend')

global_df = global_cards.merge(global_spend, how = 'inner', on = 'year', suffixes = ('_cards', '_spend'))
global_df = clean_utils.rename_columns(df = global_df, suffix = '_global')

uk_finance = read_utils.read_uk_finance()
uk_finance = clean_utils.clean_uk_finance(uk_finance)
uk_finance = calc_utils.calculate_uk_finance(uk_finance)
uk_finance = uk_finance[['year', 'cardholders','total value of purchases',"total volume of purchases"]]
uk_finance = clean_utils.rename_columns(df = uk_finance , suffix = '_uk_finance')

boe = read_utils.read_boe()
boe = clean_utils.clean_boe(boe)
boe = calc_utils.calculate_boe(boe)
boe = clean_utils.rename_columns(df = boe , suffix = '_boe')


In [ ]:
link = read_utils.read_link()

merged = visa.merge(uk_finance, how = 'outer', on = 'year')
merged = merged.merge(boe, how = 'outer', on = 'year')
merged = merged.merge(global_df, how = 'outer', on = 'year')

cardholders = merged[['year','cardholders_spoc','cardholders_uk_finance','visa_total_cards_global','total_cards_global', 'visa_marketshare_cards_global']]
cardholders = cardholders.copy()
cardholders['uk_finance_marketshare'] = cardholders['cardholders_spoc'] / cardholders['cardholders_uk_finance'] *100
cardholders['global_marketshare'] = cardholders['cardholders_spoc'] / cardholders['total_cards_global'] *100
#melt df for charts
cardholders = pd.melt(cardholders, id_vars='year',var_name='Data source', value_name='value')
cardholders = calc_utils.calculate_index(df = cardholders)

spend = merged[['year','spend_spoc', 
        'total value of purchases_uk_finance',
       'Mastercard values_boe', 'Visa Europe values_boe',
       'Mastercard and Visa values_boe', 'Visa proportion_boe',
       'debit_spend_global', 'credit_spend_global', 'visa_total_spend_global',
       'total_spend_global', 'visa_marketshare_spend_global']]
spend = spend.copy()
# #replace 2024 spending with NA
spend['spend_spoc'] = np.where(spend['year']==2024, np.nan, spend['spend_spoc'])
spend['total value of purchases_uk_finance'] = np.where(spend['year']==2024, np.nan, spend['total value of purchases_uk_finance'])
#calculate marketshare
spend['uk_finance_marketshare'] = spend['spend_spoc'] / spend['total value of purchases_uk_finance'] *100
spend['global_marketshare'] = spend['spend_spoc'] / spend['total_spend_global'] *100
spend['boe_marketshare'] = spend['spend_spoc'] / spend['Mastercard and Visa values_boe'] *100
#copy used for getting 2019 marketshare
spend_copy = spend.copy()
#melt df for charts
spend = pd.melt(spend, id_vars='year',var_name='Data source', value_name='value')
spend = calc_utils.calculate_index(df = spend)

plot_utils.plot_total_cardholders(df = cardholders)


In [ ]:
#options of marketshare threshold
# marketshare_2019 = spend_copy.iloc[0]['visa_marketshare_spend_global']
# marketshare_2019 = spend_copy.iloc[0]['global_marketshare']
marketshare_2019 = spend_copy.iloc[0]['uk_finance_marketshare']
marketshare_2019

df = merged.copy() 
#remove 2024 due to incomplete data
df = df[df['year'] != 2024]
#index spoc data
df['idx_cardholders_spoc'] = df['cardholders_spoc'].transform(lambda x: (x / x.iloc[0] * 100))
df['idx_spend_spoc'] = df['spend_spoc'].transform(lambda x: (x / x.iloc[0] * 100))
# adjust visa spend to 2019 cardholders (assume same number of cardholders each year)
df['visa_adj_spend_spoc'] = (df['spend_spoc']/df['idx_cardholders_spoc'])*100
df['total_spoc'] = df['visa_adj_spend_spoc'] / marketshare_2019 *100
#rename columns
df = df.rename(columns={'total value of purchases_uk_finance': 'total_uk_finance'})
df = df.rename(columns={'visa_total_spend_global': 'visa_total_global','total_spend_global':'total_global'})
df = df.rename(columns={'Visa Europe values_boe': 'visa_total_boe', 'Mastercard and Visa values_boe': 'total_boe'})
#filter columns
df = df[['year', 'visa_adj_spend_spoc', 'total_spoc', 'visa_total_global', 'total_global', 'total_uk_finance', 'visa_total_boe', 'total_boe' ]]
#melt df
df = pd.melt(df, id_vars='year',var_name='Data source', value_name='value')
df = calc_utils.calculate_index(df = df)

# total_uk_finance: Total spend in the UK finance sector.
# visa_total_global: Total global spend for Visa.
# total_global: Total global spend (presumably for both Visa and Mastercard).
# visa_total_boe: Total spend for Visa in the Bank of England (BOE) area.
# total_boe: Total spend in the BOE area, for both Visa and Mastercard.

In [ ]:
import pandas as pd

# Assuming 'df' is your DataFrame after transformation and melting
# Filter the DataFrame for the relevant columns (those you want to display)
data_sources = ['total_uk_finance', 'visa_total_global', 'total_global', 'visa_total_boe', 'total_boe']

# Filter rows where 'Data source' is one of the specified sources
df_filtered = df[df['Data source'].isin(data_sources)]

# Pivot the DataFrame to get each data source as columns, showing values by year
df_pivot = df_filtered.pivot_table(index='year', columns='Data source', values='value', aggfunc='first')

# Display the result
print(df_pivot)

In [ ]:
# Given data for 2021, 2022, and 2023
spend_2023 = 1.34e+12
spend_2022 = 1.17e+12
spend_2021 = 1.10e+12

# Calculate yearly growth rates
growth_rate_2022_2023 = (spend_2023 / spend_2022) - 1
growth_rate_2021_2022 = (spend_2022 / spend_2021) - 1

# Average growth rate (you could also use other techniques to refine this)
avg_growth_rate = (growth_rate_2022_2023 + growth_rate_2021_2022) / 2

# Estimate 2020 and 2019 figures based on the average growth rate
spend_2020_est = spend_2021 / (1 + avg_growth_rate)  # Backtrack from 2021 to 2020
spend_2019_est = spend_2020_est / (1 + avg_growth_rate)  # Backtrack from 2020 to 2019

# Results
print(f"Estimated Spend for 2020: {spend_2020_est:.2e}")
print(f"Estimated Spend for 2019: {spend_2019_est:.2e}")


In [ ]:
Data source     total_boe  total_global  total_uk_finance  visa_total_boe  \
year                                                                        
2019         9.02e+11      9.694000e+11      8.028740e+11    8.624950e+11   
2020         9.96e+11      9.028000e+11      8.016540e+11    8.666880e+11   
2021         1.095000e+12  9.903000e+11      8.743800e+11    9.055650e+11   
2022         1.165080e+12  1.093500e+12      9.654290e+11    8.603050e+11   
2023         1.339185e+12  1.171000e+12      1.021071e+12    8.219800e+11   

Data source  visa_total_global  
year                            
2019              8.120000e+11  
2020              7.647000e+11  
2021              7.960000e+11  
2022              7.340000e+11  
2023              6.945000e+11  

In [ ]:
   year       online_spend_ratio
0  2019           40.728857
1  2020           49.799012
2  2021           48.471550
3  2022           46.433003
4  2023           47.173528
5  2024           47.078255


In [ ]:
project_path = "/home/jupyter"
import os
import sys
sys.path.append(project_path)
from google.cloud import bigquery, storage

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

from fintrans_toolbox.src import bq_utils as bq

client = bigquery.Client()

# Summarise the data by B2B Online Abroad
UK_spending_by_country_online = '''SELECT time_period_value, destination_country, spend FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and mcc = 'All' and mcg = 'BUSINESS TO BUSINESS' and merchant_channel = 'Online' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country != 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value, spend ORDER BY time_period_value, destination_country DESC'''
df_by_country = bq.read_bq_table_sql(client, UK_spending_by_country_online)
df_by_country = df_by_country.rename(columns={'spend': 'online_spend_b2b'})
df_by_country.head()

# Extract the year from the time_period_value (assuming it's in format like '2023-Q1')
df_by_country['year'] = df_by_country['time_period_value'].str[:4].astype(int)

# Group by year and sum the online spend for each year
df_yearly_spend_total = df_by_country.groupby('year')['online_spend_b2b'].sum().reset_index()

# Optionally, sort the result by year
df_yearly_spend_total = df_yearly_spend_total.sort_values(by='year')

# Display the result
print(df_yearly_spend_total)

In [ ]:
# Summarise the data by Abroad Online All
UK_spending_by_country_online1 = '''SELECT time_period_value, destination_country, spend FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and mcg = 'All' and merchant_channel = 'Online' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country != 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value, spend ORDER BY time_period_value, destination_country DESC'''
df_by_country1 = bq.read_bq_table_sql(client, UK_spending_by_country_online1)
df_by_country1 = df_by_country1.rename(columns={'spend': 'online_spend_all'})
df_by_country1.head()
# Extract the year from the time_period_value (assuming it's in format like '2023-Q1')
df_by_country1['year'] = df_by_country1['time_period_value'].str[:4].astype(int)

# Group by year and sum the online spend for each year
df_yearly_spend_total1 = df_by_country1.groupby('year')['online_spend_all'].sum().reset_index()

# Optionally, sort the result by year
df_yearly_spend_total1 = df_yearly_spend_total1.sort_values(by='year')

# Display the result
print(df_yearly_spend_total1)

In [ ]:
#Calculate Household Online Abroad Total

# Merge the two dataframes on the 'year' column
df_yearly_spend_online_household = pd.merge(df_yearly_spend_total1, df_yearly_spend_total, on='year', how='inner')

# Create a new column for the difference between the two spending values
df_yearly_spend_online_household['spend_online_household'] = df_yearly_spend_online_household['online_spend_all'] - df_yearly_spend_online_household['online_spend_b2b']

# Optionally, keep only the relevant columns
df_yearly_spend_online_household = df_yearly_spend_online_household[['year', 'spend_online_household']]

# Display the resulting table
print(df_yearly_spend_online_household)


In [ ]:
# Summarise the data by Household Abroad B2B All
UK_spending_by_abroad_b2b = '''SELECT time_period_value, destination_country, spend FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and mcc = 'All' and mcg = 'BUSINESS TO BUSINESS' and merchant_channel = 'All' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country != 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value, spend ORDER BY time_period_value, destination_country DESC'''
df_by_country2 = bq.read_bq_table_sql(client, UK_spending_by_abroad_b2b)
df_by_country2 = df_by_country2.rename(columns={'spend': 'abroad_spend_b2b'})
df_by_country2.head()

# Extract the year from the time_period_value (assuming it's in format like '2023-Q1')
df_by_country2['year'] = df_by_country2['time_period_value'].str[:4].astype(int)

# Group by year and sum the online spend for each year
df_yearly_spend_total3 = df_by_country2.groupby('year')['abroad_spend_b2b'].sum().reset_index()

# Optionally, sort the result by year
df_yearly_spend_total3 = df_yearly_spend_total3.sort_values(by='year')

# Display the result
print(df_yearly_spend_total3)

In [ ]:
# Summarise the data by UK Abroad All Spending
UK_spending_by_abroad_all = '''SELECT time_period_value, destination_country, spend FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and mcc = 'All' and mcg = 'All' and merchant_channel = 'All' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country != 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value, spend ORDER BY time_period_value, destination_country DESC'''
df_by_country3 = bq.read_bq_table_sql(client, UK_spending_by_abroad_all)
df_by_country3 = df_by_country3.rename(columns={'spend': 'abroad_spend_all'})
df_by_country3.head()

# Extract the year from the time_period_value (assuming it's in format like '2023-Q1')
df_by_country3['year'] = df_by_country3['time_period_value'].str[:4].astype(int)

# Group by year and sum the online spend for each year
df_yearly_spend_total4 = df_by_country3.groupby('year')['abroad_spend_all'].sum().reset_index()

# Optionally, sort the result by year
df_yearly_spend_total4 = df_yearly_spend_total4.sort_values(by='year')

# Display the result
print(df_yearly_spend_total4)

In [ ]:
#Calculate Household Abroad Total

# Merge the two dataframes on the 'year' column
df_yearly_spend_abroad_household = pd.merge(df_yearly_spend_total4, df_yearly_spend_total3, on='year', how='inner')

# Create a new column for the difference between the two spending values
df_yearly_spend_abroad_household['spend_abroad_household'] = df_yearly_spend_abroad_household['abroad_spend_all'] - df_yearly_spend_abroad_household['abroad_spend_b2b']

# Optionally, keep only the relevant columns
df_yearly_spend_abroad_household = df_yearly_spend_abroad_household[['year', 'spend_abroad_household']]

# Display the resulting table
print(df_yearly_spend_abroad_household)

In [ ]:
# Calculate Household Abroad Spend Ratio
#Assuming df_yearly_spend_abroad_household and df_yearly_spend_domestic_household are already defined as dataframes with 'year' and 'spend' columns

# Merge both dataframes on 'year' column
df_combined_spend = pd.merge(df_yearly_spend_abroad_household, df_yearly_spend_domestic_household, on='year', how='inner', suffixes=('_abroad', '_domestic'))

# Calculate the Abroad Household Spend Ratio
df_combined_spend['Abroad_Household_Spend_Ratio'] = (df_combined_spend['spend_abroad_household'] / 
                                                     (df_combined_spend['spend_domestic_household'] + df_combined_spend['spend_abroad_household'])) * 100

# Optionally, keep relevant columns for the final result
df_combined_spend = df_combined_spend[['year', 'Abroad_Household_Spend_Ratio']]

# Display the resulting table
print(df_combined_spend)


In [ ]:
# Calculate Household Abroad Online Spend Ratio

# First, merge df_yearly_spend_abroad_household with df_yearly_spend_domestic_household
df_combined_spend_Online = pd.merge(df_yearly_spend_online_household, df_yearly_spend_total4, on='year', how='inner', suffixes=('_abroad', '_domestic'))

# Calculate the Abroad Household Spend Ratio
df_combined_spend_Online['Abroad_Household_Online_Spend_Ratio'] = (
    (df_yearly_spend_online_household['spend_online_household']) / 
    (df_combined_spend_Online['abroad_spend_all'])
) * 100

# Optionally, keep relevant columns for the final result
df_combined_spend_Online = df_combined_spend_Online[['year', 'Abroad_Household_Online_Spend_Ratio']]

# Display the resulting table
print(df_combined_spend_Online)


In [ ]:
Data source     total_boe  total_global  total_uk_finance  visa_total_boe  \
year                                                                        
2019         9.02e+11      9.694000e+11      8.028740e+11    8.624950e+11   
2020         9.96e+11      9.028000e+11      8.016540e+11    8.666880e+11   
2021         1.095000e+12  9.903000e+11      8.743800e+11    9.055650e+11   
2022         1.165080e+12  1.093500e+12      9.654290e+11    8.603050e+11   
2023         1.339185e+12  1.171000e+12      1.021071e+12    8.219800e+11   

Data source  visa_total_global  
year                            
2019              8.120000e+11  
2020              7.647000e+11  
2021              7.960000e+11  
2022              7.340000e+11  
2023              6.945000e+11  

In [ ]:
#Calculate Household_Abroad_Online_Spend_Ratio

# Step 1: Merge the two dataframes on the 'year' column
df_combined = pd.merge(df_combined_spend, df_combined_spend_Online, on='year', how='inner')

# Step 2: Multiply the relevant columns (the 'Abroad Household Spend Ratio' and 'Abroad Household Online Spend Ratio')
df_combined['Household_Abroad_Online_Spend_Ratio'] = df_combined['Abroad_Household_Spend_Ratio'] * df_combined['Abroad_Household_Online_Spend_Ratio']

# Step 3: Optionally, display the result
print(df_combined[['year', 'Household_Abroad_Online_Spend_Ratio']]/100)


In [ ]:
# Assume you have the additional data as a dataframe called 'df_totals'
# Example of what df_totals might look like:
df_totals = pd.DataFrame({
     'year': [2019, 2020, 2021, 2022, 2023],
     'visa_total_boe': [8.624950e+11, 8.666880e+11, 9.055650e+11, 8.603050e+11, 8.219800e+11],
     'total_boe': [9.02e+11, 9.96e+11, 1.095000e+12, 1.165080e+12, 1.339185e+12],
     'total_global': [9.694000e+11, 9.028000e+11, 9.903000e+11, 1.093500e+12, 1.171000e+12],
     'visa_total_global': [8.120000e+11, 7.647000e+11, 7.960000e+11, 7.340000e+11, 6.945000e+11],
     'total_uk_finance': [8.028740e+11, 8.016540e+11, 8.743800e+11, 9.654290e+11, 1.021071e+12]
 })

# Rename the columns in df_totals to prevent duplication
df_totals.rename(columns={
    'visa_total_boe': 'visa_total_boe_totals',
    'total_boe': 'total_boe_totals',
    'total_global': 'total_global_totals',
    'visa_total_global': 'visa_total_global_totals',
    'total_uk_finance': 'total_uk_finance_totals'
}, inplace=True)

# Now, merge the two dataframes on the 'year' column
df_combined = pd.merge(df_combined, df_totals, on='year', how='inner')

# Step 2: Multiply the `Household Abroad Online Spend Ratio` with each of the total columns
df_combined['Household_Abroad_Online_Spending_visa_total_boe'] = (df_combined['Household_Abroad_Online_Spend_Ratio'] * df_combined['visa_total_boe_totals']) / 10000
df_combined['Household_Abroad_Online_Spending_total_boe'] = (df_combined['Household_Abroad_Online_Spend_Ratio'] * df_combined['total_boe_totals']) / 10000
df_combined['Household_Abroad_Online_Spending_total_global'] = (df_combined['Household_Abroad_Online_Spend_Ratio'] * df_combined['total_global_totals']) / 10000
df_combined['Household_Abroad_Online_Spending_visa_total_global'] = (df_combined['Household_Abroad_Online_Spend_Ratio'] * df_combined['visa_total_global_totals']) / 10000
df_combined['Household_Abroad_Online_Spending_total_uk_finance'] = (df_combined['Household_Abroad_Online_Spend_Ratio'] * df_combined['total_uk_finance_totals']) / 10000

# Step 3: Display the result with the new columns for each spending category
print(df_combined[['year', 'Household_Abroad_Online_Spending_visa_total_boe', 
                   'Household_Abroad_Online_Spending_total_boe', 
                   'Household_Abroad_Online_Spending_total_global', 
                   'Household_Abroad_Online_Spending_visa_total_global', 
                   'Household_Abroad_Online_Spending_total_uk_finance']])


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Data for '_totals'
df_totals = pd.DataFrame({
     'year': [2019, 2020, 2021, 2022, 2023],
     'visa_total_boe': [8.624950e+11, 8.666880e+11, 9.055650e+11, 8.603050e+11, 8.219800e+11],
     'total_boe': [9.02e+11, 9.96e+11, 1.095000e+12, 1.165080e+12, 1.339185e+12],
     'total_global': [9.694000e+11, 9.028000e+11, 9.903000e+11, 1.093500e+12, 1.171000e+12],
     'visa_total_global': [8.120000e+11, 7.647000e+11, 7.960000e+11, 7.340000e+11, 6.945000e+11],
     'total_uk_finance': [8.028740e+11, 8.016540e+11, 8.743800e+11, 9.654290e+11, 1.021071e+12]
 })

# Create traces for the bar chart
bar_traces = []
columns = ['total_uk_finance', 'visa_total_global', 'total_global', 'visa_total_boe', 'total_boe']

for col in columns:
    bar_traces.append(go.Bar(
        x=df_totals['year'],
        y=df_totals[col] / 1e9,  # Convert values to billions
        name=col.replace('_', ' ').title(),  # Format column names for legend
        text=(df_totals[col] / 1e9).round(2),  # Show values in billions with 2 decimal places
        hoverinfo='text+x+y',  # Display info on hover
        orientation='v'
    ))

# Bar chart figure
bar_fig = go.Figure(data=bar_traces)

# Update layout for bar chart
bar_fig.update_layout(
    title="Total UK Card Spending - Bar Chart (Billions GBP)",
    xaxis_title="Year",
    yaxis_title="Amount in Billions GBP",  # Updated y-axis title
    template="plotly_dark",
    showlegend=True,
    xaxis=dict(tickmode='array', tickvals=df_totals['year']),
    yaxis=dict(
        title="Amount in Billions GBP",
        showgrid=True,
        zeroline=False,
        tickformat='.2f'  # Format y-axis to show 2 decimal places in billions
    ),
)

# Display the bar chart
bar_fig.show()


# Create traces for the line chart
line_traces = []
for col in columns:
    line_traces.append(go.Scatter(
        x=df_totals['year'],
        y=df_totals[col] / 1e9,  # Convert values to billions
        mode='lines+markers',
        name=col.replace('_', ' ').title(),  # Format column names for legend
        line=dict(width=2),  # Line thickness
        marker=dict(symbol='circle')  # Marker shape
    ))

# Line chart figure
line_fig = go.Figure(data=line_traces)

# Update layout for line chart
line_fig.update_layout(
    title="Total UK Card Spending - Line Chart (Billions GBP)",
    xaxis_title="Year",
    yaxis_title="Amount in Billions GBP",  # Updated y-axis title
    template="plotly_dark",
    showlegend=True,
    xaxis=dict(tickmode='array', tickvals=df_totals['year']),
    yaxis=dict(
        title="Amount in Billions GBP",
        showgrid=True,
        zeroline=False,
        tickformat='.2f'  # Format y-axis to show 2 decimal places in billions
    ),
)

# Display the line chart
line_fig.show()


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Data for '_totals'
df_totals = pd.DataFrame({
     'year': [2019, 2020, 2021, 2022, 2023],
     'visa_total_boe': [8.624950e+11, 8.666880e+11, 9.055650e+11, 8.603050e+11, 8.219800e+11],
     'total_boe': [9.02e+11, 9.96e+11, 1.095000e+12, 1.165080e+12, 1.339185e+12],
     'total_global': [9.694000e+11, 9.028000e+11, 9.903000e+11, 1.093500e+12, 1.171000e+12],
     'visa_total_global': [8.120000e+11, 7.647000e+11, 7.960000e+11, 7.340000e+11, 6.945000e+11],
     'total_uk_finance': [8.028740e+11, 8.016540e+11, 8.743800e+11, 9.654290e+11, 1.021071e+12]
 })

# Create traces for the bar chart
bar_traces = []
columns = ['total_uk_finance', 'visa_total_global', 'total_global', 'visa_total_boe', 'total_boe']

for col in columns:
    bar_traces.append(go.Bar(
        x=df_totals['year'],
        y=df_totals[col] / 1e12,  # Convert values to trillions
        name=col.replace('_', ' ').title(),  # Format column names for legend
        text=(df_totals[col] / 1e12).round(2),  # Show values in trillions with 2 decimal places
        hoverinfo='text+x+y',  # Display info on hover
        orientation='v'
    ))

# Bar chart figure
bar_fig = go.Figure(data=bar_traces)

# Update layout for bar chart
bar_fig.update_layout(
    title="Total UK Card Spending - Bar Chart (Trillions GBP)",
    xaxis_title="Year",
    yaxis_title="Amount in Trillions GBP",  # Updated y-axis title
    template="plotly_dark",
    showlegend=True,
    xaxis=dict(tickmode='array', tickvals=df_totals['year']),
    yaxis=dict(
        title="Amount in Trillions GBP",
        showgrid=True,
        zeroline=False,
        tickformat='.2f'  # Format y-axis to show 2 decimal places in trillions
    ),
)

# Display the bar chart
bar_fig.show()


# Create traces for the line chart
line_traces = []
for col in columns:
    line_traces.append(go.Scatter(
        x=df_totals['year'],
        y=df_totals[col] / 1e12,  # Convert values to trillions
        mode='lines+markers',
        name=col.replace('_', ' ').title(),  # Format column names for legend
        line=dict(width=2),  # Line thickness
        marker=dict(symbol='circle')  # Marker shape
    ))

# Line chart figure
line_fig = go.Figure(data=line_traces)

# Update layout for line chart
line_fig.update_layout(
    title="Total UK Card Spending - Line Chart (Trillions GBP)",
    xaxis_title="Year",
    yaxis_title="Amount in Trillions GBP",  # Updated y-axis title
    template="plotly_dark",
    showlegend=True,
    xaxis=dict(tickmode='array', tickvals=df_totals['year']),
    yaxis=dict(
        title="Amount in Trillions GBP",
        showgrid=True,
        zeroline=False,
        tickformat='.2f'  # Format y-axis to show 2 decimal places in trillions
    ),
)

# Display the line chart
line_fig.show()


In [ ]:
import pandas as pd

# Creating the DataFrame with the provided data
data = {
    'year': [2019, 2020, 2021, 2022, 2023],
    'Household_Abroad_Online_Spending_visa_total_boe': [4.180013e+10, 4.302053e+10, 3.957801e+10, 2.788121e+10, 2.748430e+10],
    'Household_Abroad_Online_Spending_total_boe': [4.371471e+10, 4.943930e+10, 4.785732e+10, 3.775852e+10, 4.477794e+10],
    'Household_Abroad_Online_Spending_total_global': [4.698120e+10, 4.481305e+10, 4.328138e+10, 3.543871e+10, 3.915438e+10],
    'Household_Abroad_Online_Spending_visa_total_global': [3.935293e+10, 3.795806e+10, 3.478943e+10, 2.378785e+10, 2.322179e+10],
    'Household_Abroad_Online_Spending_total_uk_finance': [3.891065e+10, 3.979238e+10, 3.821506e+10, 3.128812e+10, 3.414125e+10]
}

df = pd.DataFrame(data)

# Display the dataframe to confirm
df


In [ ]:
import pandas as pd

# Creating the DataFrame with the provided data and year
data = {
    'year': [2019, 2020, 2021, 2022, 2023],
    'Household_Abroad_Online_Spending_visa_total_boe': [4.180013e+10, 4.302053e+10, 3.957801e+10, 2.788121e+10, 2.748430e+10],
    'Household_Abroad_Online_Spending_total_boe': [4.371471e+10, 4.943930e+10, 4.785732e+10, 3.775852e+10, 4.477794e+10],
    'Household_Abroad_Online_Spending_total_global': [4.698120e+10, 4.481305e+10, 4.328138e+10, 3.543871e+10, 3.915438e+10],
    'Household_Abroad_Online_Spending_visa_total_global': [3.935293e+10, 3.795806e+10, 3.478943e+10, 2.378785e+10, 2.322179e+10],
    'Household_Abroad_Online_Spending_total_uk_finance': [3.891065e+10, 3.979238e+10, 3.821506e+10, 3.128812e+10, 3.414125e+10]
}

df = pd.DataFrame(data)

# Convert the values to billions
df.iloc[:, 1:] = df.iloc[:, 1:] / 1e9

# Display the DataFrame
print(df)


In [ ]:
import pandas as pd

# Creating the DataFrame with the provided data
data = {
    'year': [2019, 2020, 2021, 2022, 2023],
       'Household_Abroad_Online_Spending_visa_total_boe': [4.180013e+10, 4.302053e+10, 3.957801e+10, 2.788121e+10, 2.748430e+10],
    'Household_Abroad_Online_Spending_total_boe': [4.371471e+10, 4.943930e+10, 4.785732e+10, 3.775852e+10, 4.477794e+10],
    'Household_Abroad_Online_Spending_total_global': [4.698120e+10, 4.481305e+10, 4.328138e+10, 3.543871e+10, 3.915438e+10],
    'Household_Abroad_Online_Spending_visa_total_global': [3.935293e+10, 3.795806e+10, 3.478943e+10, 2.378785e+10, 2.322179e+10],
    'Household_Abroad_Online_Spending_total_uk_finance': [3.891065e+10, 3.979238e+10, 3.821506e+10, 3.128812e+10, 3.414125e+10]}

df = pd.DataFrame(data)

# Convert the values from trillions to billions (divide by 1e9) and format them with thousands separators
df.iloc[:, 1:] = df.iloc[:, 1:].applymap(lambda x: f"{x / 1e9:,.0f}")

# Display the updated dataframe
df



In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Define the data
data = {
    'year': [2019, 2020, 2021, 2022, 2023],
    'Household_Abroad_Online_Spending_visa_total_boe': [42, 43, 40, 28, 27],
    'Household_Abroad_Online_Spending_total_boe': [44, 49, 48, 38, 45],
    'Household_Abroad_Online_Spending_total_global': [47, 45, 43, 35, 39],
    'Household_Abroad_Online_Spending_visa_total_global': [39, 38, 35, 24, 23],
    'Household_Abroad_Online_Spending_total_uk_finance': [39, 40, 38, 31, 34]
}

# Create the DataFrame
df = pd.DataFrame(data)

# Define the columns for the bar chart
categories = [
    'Household_Abroad_Online_Spending_visa_total_boe',
    'Household_Abroad_Online_Spending_total_boe',
    'Household_Abroad_Online_Spending_total_global',
    'Household_Abroad_Online_Spending_visa_total_global',
    'Household_Abroad_Online_Spending_total_uk_finance'
]

# Create the bar traces
bar_traces = []
for category in categories:
    bar_traces.append(go.Bar(
        x=df['year'],
        y=df[category],
        name=category,
        text=df[category],  # Display value on top of bars
        hoverinfo='text+x+y',  # Display info on hover
        orientation='v'
    ))

# Plotting the Bar Chart
fig_bar = go.Figure(bar_traces)

fig_bar.update_layout(
    title="Household Abroad Online Spending by Data Source",
    barmode='group',  # Group bars by category
    xaxis_title="Year",
    yaxis_title="Spending Amount (in Billions GBP)",
    template="plotly_dark",
    xaxis=dict(tickmode='array', tickvals=df['year']),  # Ensures that x-axis is properly labeled
    yaxis=dict(
        title="Amount (in Billions GBP)",
        showgrid=True,
        zeroline=False,
        tickformat=".0f",  # Format to two decimal places
        tickprefix="£", # Optional: adds the currency symbol
    ),
    showlegend=True
)

# Display Bar Chart
fig_bar.show()



In [ ]:
# Create the line traces
line_traces = []
for category in categories:
    line_traces.append(go.Scatter(
        x=df['year'],
        y=df[category],
        mode='lines+markers',
        name=category
    ))

# Plotting the Line Chart
fig_line = go.Figure(line_traces)

fig_line.update_layout(
    title="Household Abroad Online Spending Trend by Category",
    xaxis_title="Year",
    yaxis_title="Spending Amount (in GBP)",
    template="plotly_dark",
    showlegend=True
)

# Display Line Chart
fig_line.show()


In [ ]:
import pandas as pd
import plotly.express as px

# Data for Abroad Online Spend Ratio
data_abroad_online_spend_ratio = {
    'year': [2019, 2020, 2021, 2022, 2023],
    'abroad_online_spend_ratio': [0.6677, 0.7854, 0.7546, 0.5607, 0.5413]
}

# Create dataframe
df_abroad_online_spend_ratio = pd.DataFrame(data_abroad_online_spend_ratio)

# Convert the ratio to percentage
df_abroad_online_spend_ratio['abroad_online_spend_ratio_percent'] = df_abroad_online_spend_ratio['abroad_online_spend_ratio'] * 100

# Create the Bar Chart
fig_bar = px.bar(df_abroad_online_spend_ratio, 
                 x='year', 
                 y='abroad_online_spend_ratio_percent', 
                 title='Household Abroad Online Spend Ratio (2019 - 2023)', 
                 labels={'abroad_online_spend_ratio_percent': 'Abroad Online Spend Ratio (%)', 'year': 'Year'}, 
                 color='year', 
                 text='abroad_online_spend_ratio_percent')

# Customize the bar chart
fig_bar.update_traces(texttemplate='%{text:.2f}%', textposition='outside', marker=dict(line=dict(width=1, color='black')))
fig_bar.update_layout(xaxis=dict(tickmode='linear'), yaxis=dict(range=[0, 100]))

# Show the bar chart
fig_bar.show()